In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, spearmanr, kendalltau
from sklearn import preprocessing
import scipy.stats as stats
import os


In [ ]:

def create_federated_dataset_random(num_clients=5):

    dataset = pd.read_csv('data/cox-violent-parsed_filt.csv')
    os.makedirs('federated_data_random', exist_ok=True)
    # shuffle the dataset randomly
    ds_random = dataset.sample(frac=1, random_state=42).reset_index(drop=True)
    iid_chunks = np.array_split(ds_random, num_clients)
    
    data_partitions = []

    for i, chunk in enumerate(iid_chunks):
        chunk.to_csv(f'federated_data_random/client_random_{i+1}.csv', index=False) # type: ignore
        data_partitions.append(chunk)
    
    data_partitions
    return data_partitions

def create_federated_dataset_sensitive(num_clients=5, alpha=0.5):
    """
    Split centralized dataset into multiple client datasets with different data distributions
    using Dirichlet distribution for non-IID partitioning and write data in CSV files (overwrite).
    
    Parameters:
    -----------
    dataset : pd.DataFrame
        Centralized dataset containing a 'race' column
    num_clients : int
        Number of clients to split data across
    alpha : float
        Dirichlet concentration parameter (lower = more skewed, higher = more uniform)
    output_dir : str
        Directory to save client datasets
        
    Returns:
    --------
    client_indices : list of lists
        Indices assigned to each client
    """
    # Create output directory
    output_dir = 'federated_data_sensitive'
    os.makedirs(output_dir, exist_ok=True)
    dataset = pd.read_csv('data/cox-violent-parsed_filt.csv')
    
    # Validate inputs
    if 'race' not in dataset.columns:
        raise ValueError("Dataset must contain a 'race' column")
    if num_clients < 1:
        raise ValueError("num_clients must be at least 1")
    if alpha <= 0:
        raise ValueError("alpha must be positive")
    
    # Get unique categories
    categories = dataset['race'].unique()
    client_indices = [[] for _ in range(num_clients)]
    
    # Distribute each category across clients using Dirichlet distribution
    for cat in categories:
        # Get all indices for this category
        idx_cat = dataset[dataset['race'] == cat].index.values
        np.random.shuffle(idx_cat)
        
        # Dirichlet distribution determines how many samples of this race go to each client
        # Low alpha = high skew; High alpha = more uniform
        proportions = np.random.dirichlet([alpha] * num_clients)
        
        # Convert proportions to split points
        proportions = (np.cumsum(proportions) * len(idx_cat)).astype(int)[:-1]
        split_idx = np.split(idx_cat, proportions)
        
        # Assign splits to clients
        for i in range(num_clients):
            client_indices[i].extend(split_idx[i])
    
    # Save each client's dataset
    for i in range(num_clients):
        # Shuffle local data
        client_df = dataset.iloc[client_indices[i]].sample(frac=1, random_state=42)
        
        # Save to CSV
        output_path = os.path.join(output_dir, f'client_skewed_{i+1}.csv')
        client_df.to_csv(output_path, index=False)
        
        # Print statistics
        print(f"Client {i+1} size: {len(client_df)} | Race distribution:")
        print(client_df['race'].value_counts(normalize=True))
        print()
    
    return client_indices

severity_rank = {
    "(X)": 1,
    "(NI0)": 2,
    "(CT)": 3,
    "(M03)": 4,
    "(M2)": 5,
    "(M1)": 6,
    "(CO3)": 7,
    "(TCX)": 8,
    "(F7)": 9,
    "(F6)": 10,
    "(F5)": 11,
    "(F3)": 12,
    "(F2)": 13,
    "(F1)": 14
}

def prepare_dataset(data_violent_filt):
    
    data_cleaned = data_violent_filt[['decile_score', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'c_jail_out', 'c_charge_degree', 'c_jail_in']]
    data_cleaned['c_jail_in'] = pd.to_datetime(data_cleaned['c_jail_in'], format="%d/%m/%Y %H:%M")
    data_cleaned['c_jail_out'] = pd.to_datetime(data_cleaned['c_jail_out'], format="%d/%m/%Y %H:%M")
    data_cleaned['jail_time'] = (data_cleaned['c_jail_out'] - data_cleaned['c_jail_in']).dt.total_seconds() / (3600)
    data_cleaned.drop(columns=['c_jail_in', 'c_jail_out'], inplace=True)
    # list(data_cleaned['c_charge_degree'].unique())
    data_cleaned = data_cleaned.map(lambda x: severity_rank.get(x) if x in severity_rank else x)
    data_cleaned.dropna(inplace=True)
    return data_cleaned


In [58]:

# data = create_federated_dataset_sensitive(5, 1)
# print(data)
def get_federated_datasets_random(num_clients=5):
    datasets = []
    for i in range(num_clients):
        x = pd.read_csv(f'federated_data_random/client_random_{i+1}.csv')
        datasets.append(x)
    return datasets

def get_federated_datasets_sensitive(num_clients=5):
    datasets = []
    for i in range(num_clients):
        x = pd.read_csv(f'federated_data_sensitive/client_skewed_{i+1}.csv')
        datasets.append(x)
    return datasets


In [ ]:
federated_random = get_federated_datasets_random()
federated_senditive = get_federated_datasets_sensitive()

federated_random[0]